In [4]:
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import pandas as pd

#                                        Dataset import and splitting them into training and testing.

In [2]:
heart = pd.read_csv(r"D:\VS CODE\Data Science\Machine Learning\8. Random Forest\Dataset\heart.csv").fillna(0)
print(heart.head(5))

X, y = heart.iloc[:, :-1], heart.iloc[:, -1]
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

   age  sex  cp  trestbps  chol  fbs  restecg  thalach  exang  oldpeak  slope  \
0   63    1   3       145   233    1        0      150      0      2.3      0   
1   37    1   2       130   250    0        1      187      0      3.5      0   
2   41    0   1       130   204    0        0      172      0      1.4      2   
3   56    1   1       120   236    0        1      178      0      0.8      2   
4   57    0   0       120   354    0        1      163      1      0.6      2   

   ca  thal  target  
0   0     1       1  
1   0     2       1  
2   0     2       1  
3   0     2       1  
4   0     2       1  


#                                       Selecting best hyperparameters using GridSearchCV
```js
        Use GridSearchCV when you have small dataset and parameters are not too much because GridSearchCV test each combination of hyperparameters and return the best one. But if you have time to train GridSearchCV as it takes a lot time if dataset is large, then use it.
```

In [22]:
parameters = {
                'n_estimators' : [10, 15, 20, 30],
                'max_samples'  : [0.25, 0.4, 0.5, None],
                'max_features' : [0.4, 0.5, 0.6, None],
                'random_state' : [42, 50, 60],
                # 'bootstrap' : [True, False]  # was having runtime issues for bootstrap = False. By default its True.
}

search_best_rf = GridSearchCV(estimator = RandomForestClassifier(),
                              param_grid = parameters,
                              cv = 5,  # each model in random forest will be trained 5 times.
                              n_jobs = -1)

search_best_rf.fit(X = x_train, y = y_train)

print(search_best_rf.best_params_)

{'max_features': 0.5, 'max_samples': 0.25, 'n_estimators': 20, 'random_state': 50}


#                                       Selecting best hyperparameters using RandomizedSearchCV
```js
        Use RandomizedSearchCV when you have large dataset and parameters are much and you don’t have much time to run to get the best hyperparameters value but need at least better, because RandomizedSearchCV randomly pick some combinations of hyperparameters and return the best one. But if you have time to train GridSearchCV as it takes a lot time if dataset is large, then use it.
```

In [21]:
parameters = {
                'n_estimators' : [10, 15, 20, 30],
                'max_samples'  : [0.25, 0.4, 0.5, None],
                'max_features' : [0.4, 0.5, 0.6, None],
                'random_state' : [42, 50, 60],
                # 'bootstrap' : [True, False]  # was having runtime issues for bootstrap = False. By default its True.
}

search_best_rf = RandomizedSearchCV(estimator = RandomForestClassifier(),
                                    param_distributions = parameters,
                                    cv = 5,  # each model in random forest will be trained 5 times.
                                    n_jobs = -1)

search_best_rf.fit(X = x_train, y = y_train)

print(search_best_rf.best_params_)

{'random_state': 50, 'n_estimators': 30, 'max_samples': 0.25, 'max_features': 0.4}


#                                                       OOB (Out of Bag)
```js
        Even when we do sampling with replacement(bootstrap = True) for each decision tree model inside RandomForestClassifier, there are values that bagging never touched i.e. Unseen Values. That means there are values inside x_train and y_train that RandomForestClassifier never touched|seen during training. Those values are called OOB samples.
        
        We can use these OOB values to PREDICT the accuracy ROUGHLY BEFORE WE PREDICT USING THE WHOLE TESTING DATASET. Its like getting a rough prediction before the actual prediction. The actual accuracy is around that OOB accuracy.

        Set bootstrap = True, oob_score = True and RandomForestClassifier internally performs prediction using OOB values which we can see by RandomForestClassifier_object.oob_score_.
```

In [6]:
rf = RandomForestClassifier(n_estimators = 20,
                            max_features = 0.5,
                            max_samples = 0.25,
                            random_state = 50,
                            oob_score = True)

rf.fit(X = x_train, y = y_train)
print(f"OOB Score (Rough accuracy score) = {rf.oob_score_}.")

print(f"Actual Accuracy = {accuracy_score(y_true = y_test, y_pred = rf.predict(x_test))}.")

OOB Score (Rough accuracy score) = 0.7851239669421488.
Actual Accuracy = 0.8360655737704918.
